# Airline Review Analysis - Notebook 9: Automatic Recommendations

The last step of the system is to turn the numbers into advice. For a chosen airline, the system will list its biggest problems, show the phrases passengers use when they complain, and suggest an improvement.

This is a **rule-based** system. The rules are simple "if ... then ..." conditions that I can explain, and the advice text is created from templates filled with real numbers from the data. Generating text from templates is a very simple form of natural language generation (related to Unit 4).

Input: `data/airline_topic_rates.csv`, `data/topic_priority.csv` (from Notebook 8), `data/aspect_reviews.csv`, `data/aspect_segments.csv`
Output: `data/recommendations.csv` and `data/airline_summary.csv` (used by the dashboard)

## 1. Setup and inputs

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

df = pd.read_csv("data/reviews_preprocessed.csv", usecols=["airline", "year", "recommended_bin"])
aspect_reviews = pd.read_csv("data/aspect_reviews.csv")
pieces = pd.read_csv("data/aspect_segments.csv")

airline_rates = pd.read_csv("data/airline_topic_rates.csv", index_col=0)    # complaint rate (%) per airline and topic
priority = pd.read_csv("data/topic_priority.csv", index_col=0)              # how damaging each complaint is

topics = [c for c in airline_rates.columns if c not in ("reviews", "recommend rate (%)")]
average_rate = airline_rates[topics].mean()            # average complaint rate of all airlines
damage = priority["drop (points)"]                     # drop in recommend rate when the complaint is present

print(len(airline_rates), "airlines,", len(topics), "topics")

52 airlines, 13 topics


For every review I also need to know whether it has a complaint (negative sentiment) about each topic, the same table as in Notebook 8. Adding up the complaints per airline gives the number of complaining reviews.

In [2]:
negative = aspect_reviews[aspect_reviews["label"] == "negative"].copy()
negative["complaint"] = 1
complaints = negative.pivot_table(index="review_id", columns="topic", values="complaint", fill_value=0)
complaints = complaints.reindex(index=df.index, columns=topics, fill_value=0)

complaint_counts = complaints.join(df["airline"]).groupby("airline")[topics].sum()
complaint_counts.loc[["American Airlines", "Emirates"]]

,baggage,boarding_checkin,cabin_crew,cancellation,communication,delay,entertainment,food,ground_staff,refund_compensation,seat,value_fees,wifi
airline,,,,,,,,,,,,,
American Airlines,1157.0,1499.0,722.0,1061.0,992.0,1936.0,178.0,572.0,1685.0,1101.0,957.0,1375.0,69.0
Emirates,215.0,227.0,215.0,92.0,191.0,168.0,61.0,210.0,273.0,148.0,283.0,280.0,26.0


## 2. The rules

For each airline and topic:

1. **Enough evidence:** at least 30 reviews of the airline complain about the topic. Fewer than that could be chance.
2. **Compare with other airlines:** the airline's complaint rate divided by the average complaint rate of all airlines. This gives a level:
   - **High**: at least 25% worse than the average
   - **Medium**: worse than the average
   - **Watch**: about average or better (the topic is still a problem, but not worse than at other airlines)
3. **Priority score** = complaint rate x damage. Damage is the drop in recommendation rate when a passenger complains about the topic (from Notebook 8). So a problem is a priority if it is common at this airline and also hurts a lot.
4. The five topics with the highest priority score become the recommendations.

In [3]:
MIN_COMPLAINTS = 30
HIGH_RATIO = 1.25
TOP_N = 5

## 3. Finding the phrases of the complaints

Just saying "delay is a problem" is not very useful. To make the advice specific, I find the most common word pairs (bigrams) in the negative pieces of text about that topic, like I did in Notebook 3. The pieces come from Notebook 6, and "negative" means a score below 0.35.

To keep the phrases useful, I remove stopwords (but keep "no" and "not"), lemmatize the words so that "delayed hours" and "hour delay" are counted together, and remove very general words and the words of the airline's own name.

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english")) - {"no", "not", "nor", "never", "none", "nothing", "neither", "without"}
generic = {"flight", "airline", "get", "would", "could", "also", "even", "one", "just", "still", "said", "back", "like", "say"}

lemma_cache = {}
keep_as_is = {"ground", "pass"}       # the lemmatizer would turn these into 'grind' and 'pas'

def get_lemma(word):
    if word in keep_as_is:
        return word
    if word not in lemma_cache:
        lemma = lemmatizer.lemmatize(word, pos="v")
        lemma_cache[word] = lemmatizer.lemmatize(lemma, pos="n")
    return lemma_cache[word]


def get_bigrams(text):
    text = text.lower().replace("’", "'")
    words = re.sub(r"[^a-z\s]", " ", text).split()
    words = [get_lemma(w) for w in words if w not in stop_words and len(w) > 2]
    words = [w for w in words if w not in generic]
    return sorted(set(zip(words, words[1:])))          # each pair counted once per piece


neg_pieces = pieces[pieces["learned"] < 0.35].copy()
neg_pieces["airline"] = df.loc[neg_pieces["review_id"], "airline"].values
neg_pieces["bigrams"] = neg_pieces["text"].apply(get_bigrams)
print(len(neg_pieces), "negative pieces")

177276 negative pieces


In [5]:
def key_phrases(airline, topic, n=4):
    subset = neg_pieces[(neg_pieces["airline"] == airline) & (neg_pieces["topic"] == topic)]
    name_words = set(airline.lower().split()) | {get_lemma(w) for w in airline.lower().split()} | {"airway", "airways", "air"}
    counter = Counter()
    for pairs in subset["bigrams"]:
        counter.update(p for p in pairs if not (set(p) & name_words))
    return [" ".join(p) for p, c in counter.most_common(n) if c >= 3]


def example_complaint(airline, topic, phrase=None):
    # a short, strongly negative piece of text; if a phrase is given, the piece must contain that phrase
    subset = neg_pieces[(neg_pieces["airline"] == airline) & (neg_pieces["topic"] == topic)]
    if phrase:
        pair = tuple(phrase.split(" "))
        subset = subset[subset["bigrams"].apply(lambda pairs: pair in pairs)]
    lengths = subset["text"].str.split().str.len()
    subset = subset[(lengths >= 6) & (lengths <= 30)]
    if len(subset) == 0:
        return ""
    return subset.sort_values("learned").iloc[0]["text"]


print(key_phrases("American Airlines", "delay"))
print(key_phrases("American Airlines", "communication"))
print(key_phrases("Emirates", "seat"))
print(example_complaint("American Airlines", "delay", "delay hour"))

['delay hour', 'hour delay', 'miss connect', 'wait hour']
['customer service', 'try explain', 'without explanation', 'inform not']
['seat not', 'business class', 'class seat', 'select seat']
I would stay away from this airline unless you enjoy sitting around waiting for explanations on why your 1am flight has been delayed 3 hours until 4 am


## 4. Advice templates

For every topic I wrote a short suggested action. These are general business ideas and are not learned from the data; the data decides **which** of them are shown for an airline.

In [6]:
topic_names = {"delay": "delays", "cancellation": "cancellations", "baggage": "baggage", "boarding_checkin": "boarding and check-in",
               "cabin_crew": "cabin crew", "ground_staff": "ground staff", "seat": "seats", "food": "food",
               "entertainment": "entertainment", "wifi": "WiFi", "value_fees": "value and fees",
               "refund_compensation": "refunds and compensation", "communication": "communication"}

suggestions = {
    "delay": "Improve on-time performance, for example with more realistic schedules and enough buffer time on busy routes.",
    "cancellation": "Reduce cancellations, and when they happen rebook passengers quickly with clear options.",
    "baggage": "Improve baggage handling and tracking, and make the process for lost or damaged bags simple and fast.",
    "boarding_checkin": "Make boarding and check-in smoother with better queue management, more counters and self-service options.",
    "cabin_crew": "Strengthen cabin crew training on courtesy and responsiveness, and follow up on passenger complaints about crew.",
    "ground_staff": "Train ground and customer service staff, and shorten the time it takes to answer passengers' questions and requests.",
    "seat": "Review seat comfort and legroom, especially in economy, and be clear about what each seat type offers.",
    "food": "Review the quality, temperature and variety of onboard meals and drinks.",
    "entertainment": "Update the inflight entertainment system and content, and fix broken screens quickly.",
    "wifi": "Improve the reliability and speed of onboard WiFi, and be clear about its price.",
    "value_fees": "Review fees and prices, and be transparent about baggage and extra charges so that passengers feel they get fair value.",
    "refund_compensation": "Speed up refund and compensation handling, with clear rules and status updates for the passenger.",
    "communication": "Send proactive messages (SMS or app) about delays, gate changes and cancellations, and keep passengers informed at the airport.",
}

## 5. The recommendation function

In [7]:
def recommend(airline, top_n=TOP_N):
    rows = []
    for topic in topics:
        n_complaints = complaint_counts.loc[airline, topic]
        if n_complaints < MIN_COMPLAINTS:
            continue
        rate = airline_rates.loc[airline, topic]
        ratio = rate / average_rate[topic]
        level = "High" if ratio >= HIGH_RATIO else ("Medium" if ratio >= 1.0 else "Watch")
        rows.append({"airline": airline, "topic": topic, "level": level,
                     "complaint_rate": rate, "average_rate": round(average_rate[topic], 1),
                     "complaints": int(n_complaints), "damage": damage[topic],
                     "priority_score": round(rate / 100 * damage[topic], 2)})

    if len(rows) == 0:        # some airlines have too few complaints for any topic
        return pd.DataFrame(columns=["airline", "rank", "topic", "level", "complaint_rate", "average_rate", "complaints",
                                     "damage", "priority_score", "key_phrases", "example", "topic_name", "suggestion"])

    result = pd.DataFrame(rows).sort_values("priority_score", ascending=False).head(top_n).reset_index(drop=True)
    result.insert(1, "rank", result.index + 1)
    phrases = [key_phrases(airline, t) for t in result["topic"]]
    result["key_phrases"] = [", ".join(p) for p in phrases]
    result["example"] = [example_complaint(airline, t, p[0] if p else None) for t, p in zip(result["topic"], phrases)]
    result["topic_name"] = result["topic"].map(topic_names)
    result["suggestion"] = result["topic"].map(suggestions)
    return result


def print_report(airline):
    result = recommend(airline)
    info = airline_rates.loc[airline]
    print("RECOMMENDATIONS FOR", airline.upper())
    print(int(info["reviews"]), "reviews |", info["recommend rate (%)"], "% of passengers recommend the airline")
    print()
    if len(result) == 0:
        print("No topic has at least", MIN_COMPLAINTS, "complaints, so there is not enough evidence for a recommendation.")
    for _, r in result.iterrows():
        name = r["topic_name"]
        print("Priority " + str(r["rank"]) + ": " + name.upper() + "  [" + r["level"] + "]")
        print("  " + str(round(r["complaint_rate"])) + "% of reviews complain about " + name + " (average airline: " + str(round(r["average_rate"])) + "%).")
        if r["key_phrases"]:
            print("  Frequent complaint phrases:", r["key_phrases"])
        if r["example"]:
            print('  Example: "' + r["example"] + '"')
        print("  Suggested action:", r["suggestion"])
        print()

print_report("American Airlines")

RECOMMENDATIONS FOR AMERICAN AIRLINES
5872 reviews | 12.5 % of passengers recommend the airline

Priority 1: DELAYS  [High]
  33% of reviews complain about delays (average airline: 15%).
  Frequent complaint phrases: delay hour, hour delay, miss connect, wait hour
  Example: "I would stay away from this airline unless you enjoy sitting around waiting for explanations on why your 1am flight has been delayed 3 hours until 4 am"
  Suggested action: Improve on-time performance, for example with more realistic schedules and enough buffer time on busy routes.

Priority 2: GROUND STAFF  [High]
  29% of reviews complain about ground staff (average airline: 17%).
  Frequent complaint phrases: customer service, gate agent, call customer, worst customer
  Example: "we can not do anything." Needless to say, by far the worst airline customer service we have ever encountered, and we will NEVER fly American Airlines again!"
  Suggested action: Train ground and customer service staff, and shorten the 

In [8]:
print_report("Emirates")

RECOMMENDATIONS FOR EMIRATES
1965 reviews | 44.9 % of passengers recommend the airline

Priority 1: GROUND STAFF  [Watch]
  14% of reviews complain about ground staff (average airline: 17%).
  Frequent complaint phrases: customer service, ground staff, customer care, check counter
  Example: "they have the worst customer service"
  Suggested action: Train ground and customer service staff, and shorten the time it takes to answer passengers' questions and requests.

Priority 2: VALUE AND FEES  [Watch]
  14% of reviews complain about value and fees (average airline: 17%).
  Frequent complaint phrases: business class, not worth, cost cut, low cost
  Example: "I do not recommend flying Emirates Business class, really not worth the extra money you pay"
  Suggested action: Review fees and prices, and be transparent about baggage and extra charges so that passengers feel they get fair value.

Priority 3: SEATS  [Medium]
  14% of reviews complain about seats (average airline: 12%).
  Frequent 

**Observations**

For American Airlines the system finds five problems that are all at least 25% worse than at the average airline: delays (33% of reviews against 15% on average), ground staff (29% against 17%), boarding and check-in, value and fees, and refunds. The phrases add detail, for example "delay hour" and "miss connect" for delays, "gate agent" and "customer service" for ground staff, and "not worth" and "waste money" for value.

Emirates is recommended far more often (45%), and the system reflects this: most of its top topics are labelled **Watch** (ground staff, value, boarding and baggage are below the average rate), and only seats are **Medium**. So the system does not claim serious problems where the airline is doing better than average.

## 6. One more rule: communication during disruptions

In Notebook 8 I found that poor communication makes a delay or cancellation much worse. So I add a rule for each airline: among the reviews that complain about a delay or cancellation, what share also complains about communication? If it is at least 25% higher than the average (and there are at least 100 such reviews), the system adds a special note.

In [9]:
disrupted = complaints[(complaints["delay"] == 1) | (complaints["cancellation"] == 1)].join(df["airline"])
disruption_stats = disrupted.groupby("airline")["communication"].agg(["mean", "size"])
average_share = disrupted["communication"].mean() * 100
print("average share of disruption complaints that also complain about communication:", round(average_share, 1), "%")

airline_summary = airline_rates[["reviews", "recommend rate (%)"]].copy()
airline_summary["disruption_reviews"] = disruption_stats["size"]
airline_summary["communication_share"] = (disruption_stats["mean"] * 100).round(1)
airline_summary["average_share"] = round(average_share, 1)

def special_note(row):
    if row["disruption_reviews"] >= 100 and row["communication_share"] >= 1.25 * row["average_share"]:
        return (str(row["communication_share"]) + "% of the delay and cancellation complaints also mention poor communication "
                "(average airline: " + str(row["average_share"]) + "%). Informing passengers better during disruptions should have a big effect.")
    return ""

airline_summary["special_note"] = airline_summary.apply(special_note, axis=1)
print(airline_summary[airline_summary["special_note"] != ""].shape[0], "airlines get the special note")
airline_summary[airline_summary["special_note"] != ""][["reviews", "communication_share", "average_share"]].sort_values("communication_share", ascending=False).head(8)

average share of disruption complaints that also complain about communication: 28.4 %
3 airlines get the special note


,reviews,communication_share,average_share
airline,,,
China Eastern Airlines,450,43.0,28.4
Air Europa,365,42.0,28.4
Philippine Airlines,686,39.0,28.4


**Observations**

Only three airlines get this special note: China Eastern (43% of its delay and cancellation complaints also mention communication), Air Europa (42%) and Philippine Airlines (39%), compared with an average of 28%. For these airlines, informing passengers better during disruptions is a clear first action.

## 7. Recommendations for all airlines

In [10]:
all_recommendations = pd.concat([recommend(a) for a in airline_rates.index], ignore_index=True)

all_recommendations.to_csv("data/recommendations.csv", index=False)
airline_summary.to_csv("data/airline_summary.csv")
print("saved", len(all_recommendations), "recommendations for", all_recommendations["airline"].nunique(), "airlines")

# which topics come up most often as the top priority?
all_recommendations[all_recommendations["rank"] == 1]["topic"].value_counts()

saved 245 recommendations for 49 airlines


C:\Users\Akshara\AppData\Local\Temp\ipykernel_25840\1339667294.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_recommendations = pd.concat([recommend(a) for a in airline_rates.index], ignore_index=True)


topic
ground_staff     22
value_fees       12
delay            10
baggage           3
communication     1
seat              1
Name: count, dtype: int64

**Observations**

The system produced 245 recommendations for 49 airlines. Three airlines (ANA, EVA Air and Hainan Airlines) have too few complaints for any topic to reach the minimum of 30, which is a good sign for them.

The topic that is most often the top priority is **ground staff (22 airlines)**, then **value and fees (12)** and **delay (10)**. Ground staff and value / fees were the two topics with weaker validation in Notebook 7 (they partly reflect the general mood), and their word lists are also the broadest, so the advice for those two should be read with some care.

## Summary

- The system gives advice for an airline with simple rules: enough complaints (at least 30), a comparison with the average airline (High / Medium / Watch), and a priority score = complaint rate x damage.
- The advice is made from templates filled with real numbers, the most common complaint phrases (bigrams), and an example sentence from a passenger.
- An extra rule adds a note when an airline is much worse than average at communicating during delays and cancellations.
- Results for all airlines are saved in `data/recommendations.csv` and `data/airline_summary.csv`.

Limits: the suggested actions are general ideas and not learned from data, the phrases can still contain noise, and the advice is based on what reviewers write, not on operational data like real delay times or costs.

Next: summarising what passengers say (Notebook 10) and the dashboard (`app.py`).